# 🔍 Fraud Detection — ML Lifecycle with Gemini + Flexible Data Sources

**Stack:** Google Gemini (via `google-genai`) · BigQuery · Parquet · CSV · scikit-learn · LightGBM · XGBoost · SHAP  
**Dataset:** ~700k rows · ~400 columns · binary fraud label  
**Gemini role:** Agentic advisor at every stage — profiling, preprocessing decisions, model selection, evaluation interpretation, and deployment recommendations.

---
### Supported data sources
| Source | When to use |
|---|---|
| `bigquery` | Data lives in GCP; best for large datasets and production pipelines |
| `parquet` | Local or GCS file; fastest read, preserves dtypes, ideal for 700k+ rows |
| `csv` | Local file or URL; most portable, slightly slower on large files |

Set `DATA_SOURCE` in the config cell — everything downstream is identical.

---
### Lifecycle stages
1. Setup & configuration  
2. Data ingestion (BigQuery / Parquet / CSV — auto-routed)  
3. **Gemini: Data profiling advisor**  
4. Preprocessing pipeline  
5. **Gemini: Model selection advisor**  
6. Model training (AutoML baseline + LightGBM + XGBoost)  
7. **Gemini: Evaluation interpreter**  
8. SHAP explainability  
9. Threshold tuning  
10. **Gemini: Deployment recommendations**  
11. Export to GCS / Vertex AI Model Registry

## 1. Setup & Configuration

In [ ]:
# Install dependencies (run once)
!pip install -q google-genai google-cloud-bigquery google-cloud-bigquery-storage \
    lightgbm xgboost shap imbalanced-learn scikit-learn pandas numpy matplotlib \
    google-cloud-aiplatform pyarrow fastparquet db-dtypes tqdm

In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import shap
import lightgbm as lgb
import xgboost as xgb

from google.cloud import bigquery
# google-genai is imported in the config cell after credentials are resolved

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score, classification_report,
    confusion_matrix, precision_recall_curve, roc_curve
)
from sklearn.feature_selection import mutual_info_classif
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
print('✅ Imports complete')

In [ ]:
# ── Configuration — edit these ──────────────────────────────────────────────

# DATA SOURCE — choose one: 'bigquery' | 'parquet' | 'csv'
DATA_SOURCE = 'bigquery'

# BigQuery settings (used when DATA_SOURCE = 'bigquery')
GCP_PROJECT  = 'your-gcp-project-id'
BQ_DATASET   = 'your_dataset'
BQ_TABLE     = 'fraud_transactions'

# File settings (used when DATA_SOURCE = 'parquet' or 'csv')
# Can be a local path OR a GCS URI (gs://bucket/path/file.parquet)
DATA_FILE_PATH = '/path/to/your/fraud_data.parquet'   # or .csv

# CSV-specific options (ignored for parquet/bigquery)
CSV_SEPARATOR   = ','        # delimiter character
CSV_ENCODING    = 'utf-8'    # file encoding

# Parquet-specific options
PARQUET_COLUMNS = None       # list of column names to load, or None for all
PARQUET_ENGINE  = 'pyarrow'  # 'pyarrow' or 'fastparquet'

# Shared settings
TARGET_COLUMN  = 'is_fraud'          # binary 0/1 target column name
GCS_BUCKET     = 'gs://your-bucket/fraud-model'
GEMINI_API_KEY = ''                  # leave empty to use ADC (recommended on Vertex Workbench)
GEMINI_MODEL   = 'gemini-1.5-pro'   # or 'gemini-1.5-flash' for speed
SAMPLE_FOR_EDA = 50_000             # rows for profiling (full dataset used for training)
RANDOM_STATE   = 42
# ────────────────────────────────────────────────────────────────────────────

# Configure Gemini — using the new google-genai SDK
from google import genai
from google.genai import types

if GEMINI_API_KEY:
    # API key auth — for use outside GCP (local, Colab free tier, etc.)
    gemini_client = genai.Client(api_key=GEMINI_API_KEY)
else:
    # ADC auth — zero config on Vertex AI Workbench / Colab Enterprise
    # Requires: Vertex AI API enabled + service account has roles/aiplatform.user
    gemini_client = genai.Client(
        vertexai=True,
        project=GCP_PROJECT,
        location='us-central1',   # change if your project uses a different region
    )

# Only initialise BQ client if needed
bq_client = None
if DATA_SOURCE == 'bigquery':
    from google.cloud import bigquery
    bq_client = bigquery.Client(project=GCP_PROJECT)
    print(f'✅ BigQuery client ready (project={GCP_PROJECT})')

print(f'✅ Gemini ({GEMINI_MODEL}) ready')
print(f'   Data source: {DATA_SOURCE}')

In [ ]:
# ── Gemini helper ────────────────────────────────────────────────────────────
SYSTEM_CONTEXT = (
    'You are a senior ML engineer and data scientist specializing in fraud detection.\n'
    'You are advising on a fraud dataset with ~700k rows and ~400 columns.\n'
    'Give concise, actionable, expert advice. Assume the user is technical.\n'
    'When suggesting code, use Python with LightGBM, XGBoost, scikit-learn, and BigQuery.'
)

def ask_gemini(prompt: str, context: str = '') -> str:
    """Send a prompt to Gemini via the google-genai SDK and print the response."""
    full_prompt = SYSTEM_CONTEXT
    if context:
        full_prompt += '\n\nContext:\n' + context
    full_prompt += '\n\n' + prompt

    response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=full_prompt,
        config=types.GenerateContentConfig(
            temperature=0.2,       # low temp for consistent technical advice
            max_output_tokens=1024,
        ),
    )
    text = response.text
    print('\n' + '─'*70)
    print('🤖 Gemini says:')
    print('─'*70)
    print(text)
    print('─'*70 + '\n')
    return text

print('✅ Gemini helper ready')

## 2. Data Ingestion — BigQuery / Parquet / CSV

In [ ]:
# ── Unified data loader — routes to correct backend from DATA_SOURCE ──────────
import pathlib
from tqdm import tqdm

def _resolve_gcs(path: str) -> str:
    """If path is a GCS URI, download to /tmp and return local path."""
    if not path.startswith('gs://'):
        return path
    local = '/tmp/' + path.split('/')[-1]
    print(f'  Downloading {path} → {local} ...')
    os.system(f'gsutil cp "{path}" "{local}"')
    return local

def load_data(source: str, sample_n: int = None) -> pd.DataFrame:
    """
    Load fraud dataset from BigQuery, Parquet, or CSV.
    Args:
        source   : 'bigquery' | 'parquet' | 'csv'
        sample_n : if set, return a stratified sample of this many rows
    Returns:
        pd.DataFrame
    """
    print(f'⏳ Loading data from source: {source}' + (f' (sample={sample_n:,})' if sample_n else ' (full dataset)'))

    if source == 'bigquery':
        if sample_n:
            query = f"""
                SELECT *
                FROM `{GCP_PROJECT}.{BQ_DATASET}.{BQ_TABLE}`
                WHERE RAND() < {sample_n / 700_000:.5f}
                LIMIT {sample_n}
            """
        else:
            query = f'SELECT * FROM `{GCP_PROJECT}.{BQ_DATASET}.{BQ_TABLE}`'
        df = bq_client.query(query).to_dataframe(progress_bar_type='tqdm')

    elif source == 'parquet':
        local_path = _resolve_gcs(DATA_FILE_PATH)
        df = pd.read_parquet(
            local_path,
            engine=PARQUET_ENGINE,
            columns=PARQUET_COLUMNS   # None = all columns
        )
        if sample_n and len(df) > sample_n:
            # Stratified sample preserving fraud rate
            frac = sample_n / len(df)
            df = (df.groupby(TARGET_COLUMN, group_keys=False)
                    .apply(lambda g: g.sample(frac=frac, random_state=RANDOM_STATE))
                    .reset_index(drop=True))

    elif source == 'csv':
        local_path = _resolve_gcs(DATA_FILE_PATH)
        # Read in chunks for large files to show progress
        file_size = pathlib.Path(local_path).stat().st_size if not local_path.startswith('http') else None
        chunk_size = 100_000
        chunks = []
        reader = pd.read_csv(
            local_path,
            sep=CSV_SEPARATOR,
            encoding=CSV_ENCODING,
            chunksize=chunk_size,
            low_memory=False
        )
        for chunk in tqdm(reader, desc='Reading CSV chunks'):
            chunks.append(chunk)
        df = pd.concat(chunks, ignore_index=True)
        if sample_n and len(df) > sample_n:
            frac = sample_n / len(df)
            df = (df.groupby(TARGET_COLUMN, group_keys=False)
                    .apply(lambda g: g.sample(frac=frac, random_state=RANDOM_STATE))
                    .reset_index(drop=True))

    else:
        raise ValueError(f'Unknown DATA_SOURCE: "{source}". Use bigquery | parquet | csv')

    print(f'✅ Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
    mem = df.memory_usage(deep=True).sum()
    print(f'   Memory: {mem / 1e6:.1f} MB  |  Fraud rate: {df[TARGET_COLUMN].mean()*100:.3f}%')
    return df

print('✅ load_data() ready')

In [ ]:
# ── Load sample for EDA / profiling ──────────────────────────────────────────
df_sample = load_data(DATA_SOURCE, sample_n=SAMPLE_FOR_EDA)
df_sample.head(3)

In [ ]:
# ── Load full dataset for training ───────────────────────────────────────────
# For very large CSVs on low-RAM machines, consider loading parquet instead
# or increasing SAMPLE_FOR_EDA and skipping this cell.
df = load_data(DATA_SOURCE, sample_n=None)
print(f'Full dataset memory: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB')

## 3. 🤖 Gemini: Data Profiling Advisor

In [ ]:
# Compute profile statistics
fraud_rate = df_sample[TARGET_COLUMN].mean() * 100
null_rates = (df_sample.isnull().mean() * 100).sort_values(ascending=False)
high_null_cols = null_rates[null_rates > 20]
dtype_counts = df_sample.dtypes.value_counts().to_dict()
cardinality = df_sample.select_dtypes('object').nunique().sort_values(ascending=False).head(10)
numeric_cols = df_sample.select_dtypes(include=np.number).columns.tolist()

print(f'Fraud rate:          {fraud_rate:.3f}%')
print(f'Columns with >20% null: {len(high_null_cols)}')
print(f'Dtypes: {dtype_counts}')
print(f'Numeric columns: {len(numeric_cols)}')

# Visualise class balance
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df_sample[TARGET_COLUMN].value_counts().plot(kind='bar', ax=axes[0], color=['#378ADD','#E24B4A'])
axes[0].set_title('Class distribution'); axes[0].set_xlabel('')

null_rates.head(30).plot(kind='bar', ax=axes[1], color='#7F77DD')
axes[1].set_title('Top 30 columns by null %'); axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())

cardinality.plot(kind='bar', ax=axes[2], color='#1D9E75')
axes[2].set_title('Top categorical cardinalities')
plt.tight_layout(); plt.show()

In [ ]:
profile_context = f"""
Dataset shape: {df_sample.shape[0]:,} rows (sample), {df_sample.shape[1]} columns
Fraud rate: {fraud_rate:.3f}%
Columns with >20% nulls: {len(high_null_cols)} columns
Top null columns: {high_null_cols.head(5).to_dict()}
Dtype breakdown: {dtype_counts}
Top categorical cardinalities: {cardinality.head(5).to_dict()}
Numeric columns count: {len(numeric_cols)}
"""

_ = ask_gemini(
    prompt="""Analyse this fraud dataset profile. Tell me:
1. How severe is the class imbalance and what strategy should I use?
2. What should I do with high-null columns?
3. Any immediate red flags from the cardinality or dtype breakdown?
4. What feature engineering opportunities do you see?
Be specific and prioritised.""",
    context=profile_context
)

## 4. Preprocessing Pipeline

In [ ]:
# ── Step 4a: Feature selection via mutual information ────────────────────────
print('⏳ Computing mutual information scores (this may take a few minutes)...')

# Work on sample for speed
X_sample = df_sample.drop(columns=[TARGET_COLUMN])
y_sample = df_sample[TARGET_COLUMN]

# Keep only numeric for MI (encode categoricals separately)
X_num = X_sample.select_dtypes(include=np.number).fillna(0)
mi_scores = mutual_info_classif(X_num, y_sample, random_state=RANDOM_STATE)
mi_series = pd.Series(mi_scores, index=X_num.columns).sort_values(ascending=False)

# Keep features with MI > threshold
MI_THRESHOLD = 0.001
selected_features = mi_series[mi_series > MI_THRESHOLD].index.tolist()
print(f'✅ Selected {len(selected_features)} / {len(X_num.columns)} numeric features (MI > {MI_THRESHOLD})')

# Plot top features
mi_series.head(30).plot(kind='barh', figsize=(10, 8), color='#378ADD')
plt.title('Top 30 features by mutual information with fraud label')
plt.xlabel('Mutual information score')
plt.gca().invert_yaxis()
plt.tight_layout(); plt.show()

In [ ]:
# ── Step 4b: Full preprocessing on training data ─────────────────────────────
from sklearn.model_selection import train_test_split

# Drop columns with >50% nulls
null_rates_full = (df.isnull().mean() * 100)
drop_cols = null_rates_full[null_rates_full > 50].index.tolist()
print(f'Dropping {len(drop_cols)} columns with >50% nulls')
df_clean = df.drop(columns=drop_cols)

# Encode categoricals
cat_cols = df_clean.select_dtypes('object').columns.tolist()
for col in cat_cols:
    df_clean[col] = LabelEncoder().fit_transform(df_clean[col].astype(str))
print(f'Encoded {len(cat_cols)} categorical columns')

# Impute remaining nulls with median
df_clean = df_clean.fillna(df_clean.median(numeric_only=True))

# Restrict to selected features + target
available_selected = [f for f in selected_features if f in df_clean.columns]
X = df_clean[available_selected]
y = df_clean[TARGET_COLUMN]

# Time-based split (use last 20% as test if you have a timestamp column)
# If no timestamp, use stratified random split:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.15, stratify=y_train, random_state=RANDOM_STATE
)

print(f'\nTrain: {X_train.shape[0]:,} rows  |  Val: {X_val.shape[0]:,}  |  Test: {X_test.shape[0]:,}')
print(f'Train fraud rate: {y_train.mean()*100:.3f}%')
print(f'Test  fraud rate: {y_test.mean()*100:.3f}%')

In [ ]:
# ── Step 4c: Handle class imbalance ──────────────────────────────────────────
# Strategy depends on fraud rate:
#   < 1%  → undersample majority + SMOTE minority (combined)
#   1-5%  → class_weight='balanced' or scale_pos_weight in tree models
#   > 5%  → class_weight sufficient

FRAUD_RATE = y_train.mean()
print(f'Train fraud rate: {FRAUD_RATE*100:.3f}%')

if FRAUD_RATE < 0.01:
    print('Applying combined SMOTE + RandomUnderSampler...')
    pipeline = ImbPipeline([
        ('under', RandomUnderSampler(sampling_strategy=0.1, random_state=RANDOM_STATE)),
        ('over',  SMOTE(sampling_strategy=0.5, random_state=RANDOM_STATE, n_jobs=-1))
    ])
    X_train_res, y_train_res = pipeline.fit_resample(X_train, y_train)
    SCALE_POS_WEIGHT = 1.0  # already balanced
else:
    print('Using scale_pos_weight in tree models (no resampling needed)')
    X_train_res, y_train_res = X_train, y_train
    SCALE_POS_WEIGHT = (1 - FRAUD_RATE) / FRAUD_RATE  # for XGBoost

print(f'Resampled train shape: {X_train_res.shape}')
print(f'Resampled fraud rate:  {y_train_res.mean()*100:.2f}%')
print(f'scale_pos_weight:      {SCALE_POS_WEIGHT:.1f}')

## 5. 🤖 Gemini: Model Selection Advisor

In [ ]:
model_context = f"""
After preprocessing:
- Training rows: {X_train_res.shape[0]:,}
- Features selected: {X_train_res.shape[1]}
- Original fraud rate: {FRAUD_RATE*100:.3f}%
- Resampled fraud rate: {y_train_res.mean()*100:.2f}%
- scale_pos_weight for XGBoost: {SCALE_POS_WEIGHT:.1f}
- Evaluation metric priority: AUC-PR > AUC-ROC > F1 (due to imbalance)
"""

_ = ask_gemini(
    prompt="""Given this preprocessed fraud dataset, recommend:
1. Which model to try first and why (LightGBM vs XGBoost vs Logistic Regression)
2. Exact hyperparameter starting values for LightGBM for fraud detection
3. What cross-validation strategy to use
4. What early stopping settings
5. One ensemble or stacking approach worth trying if single models plateau""",
    context=model_context
)

## 6. Model Training

In [ ]:
# ── Model A: LightGBM ────────────────────────────────────────────────────────
print('Training LightGBM...')

lgbm_params = {
    'objective':        'binary',
    'metric':           ['auc', 'average_precision'],
    'boosting_type':    'gbdt',
    'num_leaves':       127,
    'learning_rate':    0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq':     5,
    'min_child_samples': 20,
    'scale_pos_weight': SCALE_POS_WEIGHT,
    'n_jobs':           -1,
    'verbose':          -1,
    'random_state':     RANDOM_STATE,
}

dtrain_lgb = lgb.Dataset(X_train_res, label=y_train_res)
dval_lgb   = lgb.Dataset(X_val,       label=y_val, reference=dtrain_lgb)

callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=True),
    lgb.log_evaluation(period=100)
]

lgbm_model = lgb.train(
    lgbm_params,
    dtrain_lgb,
    num_boost_round=1000,
    valid_sets=[dtrain_lgb, dval_lgb],
    valid_names=['train', 'val'],
    callbacks=callbacks
)

lgbm_val_pred = lgbm_model.predict(X_val)
lgbm_auc_roc = roc_auc_score(y_val, lgbm_val_pred)
lgbm_auc_pr  = average_precision_score(y_val, lgbm_val_pred)
print(f'\n✅ LightGBM  |  Val AUC-ROC: {lgbm_auc_roc:.4f}  |  Val AUC-PR: {lgbm_auc_pr:.4f}')

In [ ]:
# ── Model B: XGBoost ─────────────────────────────────────────────────────────
print('Training XGBoost...')

xgb_params = {
    'objective':        'binary:logistic',
    'eval_metric':      ['auc', 'aucpr'],
    'max_depth':        6,
    'eta':              0.05,
    'subsample':        0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 10,
    'scale_pos_weight': SCALE_POS_WEIGHT,
    'tree_method':      'hist',   # fast on large datasets; use 'gpu_hist' if GPU available
    'nthread':          -1,
    'seed':             RANDOM_STATE,
}

dtrain_xgb = xgb.DMatrix(X_train_res, label=y_train_res)
dval_xgb   = xgb.DMatrix(X_val,       label=y_val)

xgb_model = xgb.train(
    xgb_params,
    dtrain_xgb,
    num_boost_round=1000,
    evals=[(dtrain_xgb, 'train'), (dval_xgb, 'val')],
    early_stopping_rounds=50,
    verbose_eval=100
)

xgb_val_pred = xgb_model.predict(dval_xgb)
xgb_auc_roc  = roc_auc_score(y_val, xgb_val_pred)
xgb_auc_pr   = average_precision_score(y_val, xgb_val_pred)
print(f'\n✅ XGBoost  |  Val AUC-ROC: {xgb_auc_roc:.4f}  |  Val AUC-PR: {xgb_auc_pr:.4f}')

In [ ]:
# ── Model C: Logistic Regression baseline ────────────────────────────────────
print('Training Logistic Regression baseline...')
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_res)
X_val_scaled   = scaler.transform(X_val)

lr_model = LogisticRegression(
    class_weight='balanced', max_iter=500, solver='saga', n_jobs=-1, random_state=RANDOM_STATE
)
lr_model.fit(X_train_scaled, y_train_res)
lr_val_pred = lr_model.predict_proba(X_val_scaled)[:, 1]
lr_auc_roc  = roc_auc_score(y_val, lr_val_pred)
lr_auc_pr   = average_precision_score(y_val, lr_val_pred)
print(f'\n✅ Logistic Regression  |  Val AUC-ROC: {lr_auc_roc:.4f}  |  Val AUC-PR: {lr_auc_pr:.4f}')

# Summary table
results = pd.DataFrame({
    'Model':   ['LightGBM', 'XGBoost', 'Logistic Regression'],
    'AUC-ROC': [lgbm_auc_roc, xgb_auc_roc, lr_auc_roc],
    'AUC-PR':  [lgbm_auc_pr,  xgb_auc_pr,  lr_auc_pr],
})
print('\n', results.sort_values('AUC-PR', ascending=False).to_string(index=False))

## 7. 🤖 Gemini: Evaluation Interpreter

In [ ]:
# Plot ROC and PR curves for all models
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, pred in [('LightGBM', lgbm_val_pred), ('XGBoost', xgb_val_pred), ('LR baseline', lr_val_pred)]:
    fpr, tpr, _ = roc_curve(y_val, pred)
    prec, rec, _ = precision_recall_curve(y_val, pred)
    axes[0].plot(fpr, tpr, label=f'{name} ({roc_auc_score(y_val,pred):.3f})')
    axes[1].plot(rec, prec, label=f'{name} ({average_precision_score(y_val,pred):.3f})')

axes[0].plot([0,1],[0,1],'k--'); axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('ROC curve'); axes[0].legend()
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall curve'); axes[1].legend()
plt.tight_layout(); plt.show()

# Pick best model
best_model_name = results.sort_values('AUC-PR', ascending=False).iloc[0]['Model']
print(f'\n🏆 Best model by AUC-PR: {best_model_name}')

In [ ]:
eval_context = results.to_string(index=False)

_ = ask_gemini(
    prompt="""Interpret these validation metrics for the 3 models on a fraud dataset.
1. Which model should I promote to test set evaluation and why?
2. Is the AUC-PR gap between models meaningful or noise?
3. What additional evaluation steps should I do before selecting the final model?
4. Should I consider ensembling the top 2 models?
5. What AUC-PR threshold would you consider 'production-ready' for fraud?""",
    context=eval_context
)

In [ ]:
# Final evaluation on held-out test set (run once, on the best model)
# Using LightGBM as example — swap for best model
test_pred = lgbm_model.predict(X_test)

test_auc_roc = roc_auc_score(y_test, test_pred)
test_auc_pr  = average_precision_score(y_test, test_pred)

THRESHOLD = 0.5  # will be tuned below
test_labels = (test_pred >= THRESHOLD).astype(int)
cm = confusion_matrix(y_test, test_labels)

print(f'Test AUC-ROC: {test_auc_roc:.4f}')
print(f'Test AUC-PR:  {test_auc_pr:.4f}')
print(f'\nConfusion matrix (threshold={THRESHOLD}):')
print(pd.DataFrame(cm, index=['Actual 0','Actual 1'], columns=['Pred 0','Pred 1']))
print('\n', classification_report(y_test, test_labels, target_names=['legit','fraud']))

## 8. SHAP Explainability

In [ ]:
print('⏳ Computing SHAP values (sampling 5k rows for speed)...')
shap_sample = X_test.sample(5000, random_state=RANDOM_STATE)

explainer   = shap.TreeExplainer(lgbm_model)
shap_values = explainer.shap_values(shap_sample)

# For binary classification, shap_values is a list [class0, class1]
sv = shap_values[1] if isinstance(shap_values, list) else shap_values

plt.figure(figsize=(10, 8))
shap.summary_plot(sv, shap_sample, max_display=20, show=False)
plt.title('SHAP feature importance — top 20 fraud drivers')
plt.tight_layout(); plt.show()

# Bar chart of mean |SHAP|
mean_shap = pd.Series(np.abs(sv).mean(axis=0), index=shap_sample.columns)
mean_shap.sort_values(ascending=False).head(20).plot(
    kind='barh', figsize=(10, 6), color='#D85A30'
)
plt.title('Mean |SHAP| — top 20 features')
plt.gca().invert_yaxis()
plt.tight_layout(); plt.show()

## 9. Threshold Tuning

In [ ]:
# ── Business-driven threshold tuning ─────────────────────────────────────────
# Adjust cost_fn / cost_fp to match your business context:
#   cost_fn = cost of missing a fraud (false negative)
#   cost_fp = cost of incorrectly flagging a legit transaction (false positive)
COST_FN = 100   # e.g. $100 average fraud loss per missed case
COST_FP = 5     # e.g. $5 for manual review per false alert

prec_arr, rec_arr, thresh_arr = precision_recall_curve(y_val, lgbm_val_pred)
thresholds = thresh_arr

costs = []
n_fraud = y_val.sum()
n_legit = len(y_val) - n_fraud

for t in thresholds:
    preds = (lgbm_val_pred >= t).astype(int)
    cm_t  = confusion_matrix(y_val, preds)
    tn, fp, fn, tp = cm_t.ravel()
    total_cost = fn * COST_FN + fp * COST_FP
    costs.append({'threshold': t, 'total_cost': total_cost, 'fn': fn, 'fp': fp, 'tp': tp})

cost_df = pd.DataFrame(costs)
optimal_row = cost_df.loc[cost_df['total_cost'].idxmin()]
OPTIMAL_THRESHOLD = optimal_row['threshold']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(cost_df['threshold'], cost_df['total_cost'])
axes[0].axvline(OPTIMAL_THRESHOLD, color='red', linestyle='--', label=f'Optimal: {OPTIMAL_THRESHOLD:.3f}')
axes[0].set_xlabel('Threshold'); axes[0].set_ylabel('Total business cost')
axes[0].set_title('Business cost vs threshold'); axes[0].legend()

axes[1].plot(cost_df['threshold'], cost_df['fp'], label='False positives', color='orange')
axes[1].plot(cost_df['threshold'], cost_df['fn'], label='False negatives', color='red')
axes[1].axvline(OPTIMAL_THRESHOLD, color='gray', linestyle='--')
axes[1].set_xlabel('Threshold'); axes[1].set_title('FP vs FN by threshold'); axes[1].legend()
plt.tight_layout(); plt.show()

print(f'\n🎯 Optimal threshold: {OPTIMAL_THRESHOLD:.3f}')
print(f'   At this threshold → FP: {int(optimal_row["fp"]):,}  |  FN: {int(optimal_row["fn"]):,}')
print(f'   Total cost: ${optimal_row["total_cost"]:,.0f}')

## 10. 🤖 Gemini: Deployment Recommendations

In [ ]:
deploy_context = f"""
Final model: LightGBM
Test AUC-ROC: {test_auc_roc:.4f}
Test AUC-PR:  {test_auc_pr:.4f}
Optimal threshold: {OPTIMAL_THRESHOLD:.3f}
Cost config: false_negative=$100, false_positive=$5
Top 5 SHAP features: {mean_shap.sort_values(ascending=False).head(5).index.tolist()}
Target infra: Vertex AI Endpoints (GCP)
"""

_ = ask_gemini(
    prompt="""Given these final model metrics and configuration:
1. Is this model ready for production deployment on Vertex AI? What's your confidence?
2. What monitoring metrics should I set up in Vertex AI Model Monitoring?
3. What data drift signals should trigger a retraining pipeline?
4. Recommended retraining cadence for fraud models?
5. Any final risks or gotchas before going live?""",
    context=deploy_context
)

## 11. Export to GCS & Vertex AI Model Registry

In [ ]:
import joblib
from google.cloud import aiplatform
from datetime import datetime

aiplatform.init(project=GCP_PROJECT, location='us-central1')

# ── Save artefacts locally first ──────────────────────────────────────────────
MODEL_DIR = '/tmp/fraud_model'
os.makedirs(MODEL_DIR, exist_ok=True)

lgbm_model.save_model(f'{MODEL_DIR}/lgbm_fraud.txt')
joblib.dump(scaler,             f'{MODEL_DIR}/scaler.pkl')
joblib.dump(selected_features,  f'{MODEL_DIR}/selected_features.pkl')
joblib.dump({'threshold': OPTIMAL_THRESHOLD,
             'fraud_rate': float(FRAUD_RATE),
             'auc_pr': test_auc_pr,
             'auc_roc': test_auc_roc,
             'trained_at': datetime.utcnow().isoformat()},
            f'{MODEL_DIR}/metadata.pkl')

# ── Upload to GCS ─────────────────────────────────────────────────────────────
!gsutil -m cp -r /tmp/fraud_model {GCS_BUCKET}/
print(f'✅ Model artefacts uploaded to {GCS_BUCKET}/fraud_model')

# ── Register in Vertex AI Model Registry ─────────────────────────────────────
# Note: requires a serving container. Using pre-built LightGBM container.
model = aiplatform.Model.upload(
    display_name='fraud-lgbm-v1',
    artifact_uri=f'{GCS_BUCKET}/fraud_model',
    serving_container_image_uri='us-docker.pkg.dev/vertex-ai/prediction/sklearn-cpu.1-3:latest',
    description=f'LightGBM fraud model | AUC-PR={test_auc_pr:.4f} | threshold={OPTIMAL_THRESHOLD:.3f}',
    labels={'env': 'dev', 'fraud_rate': str(round(FRAUD_RATE*100, 2))}
)
print(f'✅ Model registered: {model.resource_name}')
print(f'   View in console: https://console.cloud.google.com/vertex-ai/models')

In [ ]:
# ── (Optional) Deploy to Vertex AI Endpoint ───────────────────────────────────
# Uncomment when ready to serve live traffic

# endpoint = aiplatform.Endpoint.create(
#     display_name='fraud-detection-endpoint',
#     labels={'env': 'dev'}
# )
# model.deploy(
#     endpoint=endpoint,
#     deployed_model_display_name='fraud-lgbm-v1',
#     machine_type='n1-standard-4',
#     min_replica_count=1,
#     max_replica_count=5,
#     traffic_percentage=100
# )
# print(f'✅ Model deployed to endpoint: {endpoint.resource_name}')
# print(f'   Predict with: endpoint.predict(instances=[{{...}}])')

## Summary

| Stage | Tool |
|---|---|
| Data ingestion | BigQuery · Parquet · CSV (set `DATA_SOURCE`) |
| Feature selection | Mutual information (sklearn) |
| Class imbalance | SMOTE + undersampling (imbalanced-learn) |
| Model training | LightGBM · XGBoost · Logistic Regression |
| Hyperparameter tuning | Manual → next step: Vertex AI Hyperparameter Tuning |
| Explainability | SHAP TreeExplainer |
| Threshold tuning | Business cost matrix |
| Agentic advisor | Gemini 1.5 Pro at every lifecycle stage |
| Artefact storage | GCS |
| Model registry | Vertex AI Model Registry |
| Serving | Vertex AI Endpoint (optional, see cell above) |

---
### Data source quick-reference

**BigQuery** — best for production and large datasets already in GCP:
```python
DATA_SOURCE  = 'bigquery'
GCP_PROJECT  = 'my-project'
BQ_DATASET   = 'fraud_ds'
BQ_TABLE     = 'transactions'
```

**Parquet (local or GCS)** — fastest reads, preserves dtypes, ideal for 700k+ rows:
```python
DATA_SOURCE    = 'parquet'
DATA_FILE_PATH = '/data/fraud.parquet'          # local
DATA_FILE_PATH = 'gs://my-bucket/fraud.parquet' # GCS — auto-downloaded
PARQUET_COLUMNS = ['col_a', 'col_b', 'is_fraud'] # optional column subset
```

**CSV (local, GCS, or URL)** — most portable; chunked reading handles large files:
```python
DATA_SOURCE    = 'csv'
DATA_FILE_PATH = '/data/fraud.csv'
CSV_SEPARATOR  = ','       # use '\t' for TSV
CSV_ENCODING   = 'utf-8'
```

---
**Next steps:**
- Convert this notebook into a **Vertex AI Pipeline** (Kubeflow) for automated retraining
- Enable **Vertex AI Model Monitoring** on the endpoint for drift detection
- Add **Vertex AI Hyperparameter Tuning** job for automated HPO at scale